In [4]:
import cv2
import csv
import os
import math
import torch
import numpy as np
from collections import defaultdict
from ultralytics import YOLO

# CONFIGURACION DE BYTETRACK (SIN RETRASO)
# Creamos un archivo para que los objetos desaparezcan rápido si salen del video
TRACKER_CFG = 'bytetrack_rapido.yaml'
with open(TRACKER_CFG, 'w') as f:
    f.write("""
tracker_type: bytetrack
track_high_thresh: 0.35
track_low_thresh: 0.1
new_track_thresh: 0.4
track_buffer: 8
match_thresh: 0.8
fuse_score: True
""".strip())

# CONFIGURACION DE RUTAS Y DISPOSITIVO
VIDEO_ENTRADA = 'C0142.MP4'
VIDEO_SALIDA = 'resultado_C0142.mp4'
CSV_SALIDA = 'detecciones_seguimiento.csv'

MODELO_GENERAL_PATH = 'yolo11n.pt'
MODELO_MATRICULAS_PATH = 'best.pt'

CLASES_INTERES = [0, 2, 3, 5, 7]      # persona, coche, moto, guagua, camion
CLASES_VEHICULOS = [2, 3, 5, 7]

DISPOSITIVO = 0 if torch.cuda.is_available() else 'cpu'
print(f"Dispositivo asignado: {DISPOSITIVO}")

# CARGA DE MODELOS
modelo_general = YOLO(MODELO_GENERAL_PATH)
modelo_matriculas = YOLO(MODELO_MATRICULAS_PATH)

# VIDEO Y CSV
cap = cv2.VideoCapture(VIDEO_ENTRADA)
if not cap.isOpened():
    raise FileNotFoundError(f"No se pudo abrir el video: {VIDEO_ENTRADA}")

ancho = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
alto = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
fps_salida = fps if fps > 0 else 25.0
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

out = cv2.VideoWriter(
    VIDEO_SALIDA,
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps_salida,
    (ancho, alto)
)

f_csv = open(CSV_SALIDA, mode='w', newline='', encoding='utf-8')
csv_writer = csv.writer(f_csv)

# CAMBIO APLICADO: Las 15 columnas exactas que pide la practica en el CSV
csv_writer.writerow([
    'fotograma', 'tipo_objeto', 'confianza', 'identificador_tracking', 
    'x1', 'y1', 'x2', 'y2', 
    'matricula_en_su_caso', 'confianza_matricula', 
    'mx1', 'my1', 'mx2', 'my2', 'texto_matricula'
])

MIN_FRAMES_VALIDO = 10
MIN_DESPLAZAMIENTO = 40

historial_tracks = defaultdict(lambda: {
    'inicio': None,
    'actual': None,
    'frames': 0,
    'clases': defaultdict(int)
})

conteo_unicos = {
    'person': set(),
    'car': set(),
    'motorcycle': set(),
    'bus': set(),
    'truck': set()
}

# Memoria corta para guardar datos de las matriculas
cache_matriculas = {}

COLORES = {
    'person': (255, 105, 65),     # Azul/Cian
    'car': (40, 200, 40),         # Verde
    'motorcycle': (0, 165, 255),  # Naranja
    'bus': (200, 200, 0),         # Celeste
    'truck': (180, 0, 180),       # Magenta
    'matricula': (0, 255, 255)    # Amarillo
}

print(f"Procesando {total_frames} fotogramas...")

# BUCLE PRINCIPAL
frame_idx = 0

with torch.no_grad():
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        frame_idx += 1

        resultados_track = modelo_general.track(
            source=frame,
            persist=True,
            classes=CLASES_INTERES,
            tracker=TRACKER_CFG,
            conf=0.35,
            imgsz=640,
            device=DISPOSITIVO,
            verbose=False
        )

        if resultados_track and len(resultados_track) > 0 and resultados_track[0].boxes is not None:
            cajas = resultados_track[0].boxes

            for box in cajas:
                x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
                confianza = float(box.conf[0])
                clase_id = int(box.cls[0])
                nombre_clase = modelo_general.names[clase_id]
                track_id = int(box.id[0]) if box.id is not None else -1

                # Ignorar cosas muy pequeñas a lo lejos
                w_box = x2 - x1
                h_box = y2 - y1
                if w_box < 25 or h_box < 25:
                    continue

                cx = int((x1 + x2) / 2)
                cy = int((y1 + y2) / 2)

                # Buscar la clase que mas veces ha salido (por si YOLO se equivoca un frame)
                clase_dominante = nombre_clase
                if track_id != -1:
                    t = historial_tracks[track_id]
                    t['frames'] += 1
                    t['clases'][nombre_clase] += 1
                    if t['inicio'] is None:
                        t['inicio'] = (cx, cy)
                    t['actual'] = (cx, cy)

                    clase_dominante = max(t['clases'].items(), key=lambda x: x[1])[0]

                    # Conteo original intacto
                    if t['frames'] >= MIN_FRAMES_VALIDO:
                        disp = math.hypot(t['actual'][0] - t['inicio'][0], t['actual'][1] - t['inicio'][1])
                        es_vehiculo = clase_id in CLASES_VEHICULOS
                        
                        if (not es_vehiculo and disp >= 20) or (es_vehiculo and disp >= MIN_DESPLAZAMIENTO):
                            for c in conteo_unicos:
                                conteo_unicos[c].discard(track_id)
                            if clase_dominante in conteo_unicos:
                                conteo_unicos[clase_dominante].add(track_id)

                # Variables vacias para la matricula
                caja_matricula = None
                conf_matricula = None

                # BUSCAR MATRICULAS
                if clase_id in CLASES_VEHICULOS:
                    vx1, vy1 = max(0, x1), max(0, y1)
                    vx2, vy2 = min(ancho, x2), min(alto, y2)
                    w_veh = vx2 - vx1
                    h_veh = vy2 - vy1

                    # Ver si el coche esta tocando el borde izquierdo o derecho
                    esta_saliendo = (vx1 <= 15) or (vx2 >= ancho - 15)

                    if w_veh > 50 and h_veh > 50 and not esta_saliendo:
                        usar_detector = True

                        # Mirar si ya vimos la matricula hace poco (menos de 4 frames)
                        if track_id != -1 and track_id in cache_matriculas:
                            frames_pasados = frame_idx - cache_matriculas[track_id]['frame']
                            if frames_pasados <= 4:
                                rx1, ry1, rx2, ry2 = cache_matriculas[track_id]['rel_coords']
                                gx1 = int(vx1 + rx1 * w_veh)
                                gy1 = int(vy1 + ry1 * h_veh)
                                gx2 = int(vx1 + rx2 * w_veh)
                                gy2 = int(vy1 + ry2 * h_veh)
                                caja_matricula = (gx1, gy1, gx2, gy2)
                                
                                # Recuperar la confianza que habiamos guardado
                                conf_matricula = cache_matriculas[track_id].get('conf', 0.0)
                                usar_detector = False

                        if usar_detector:
                            roi_vehiculo = frame[vy1:vy2, vx1:vx2]
                            res_placas = modelo_matriculas(
                                roi_vehiculo,
                                imgsz=320,
                                conf=0.35,
                                device=DISPOSITIVO,
                                verbose=False
                            )

                            if res_placas and len(res_placas) > 0 and len(res_placas[0].boxes) > 0:
                                mejor_placa = res_placas[0].boxes[0]
                                px1_r, py1_r, px2_r, py2_r = map(int, mejor_placa.xyxy[0].tolist())
                                
                                gx1 = vx1 + px1_r
                                gy1 = vy1 + py1_r
                                gx2 = vx1 + px2_r
                                gy2 = vy1 + py2_r
                                caja_matricula = (gx1, gy1, gx2, gy2)
                                conf_matricula = float(mejor_placa.conf[0])

                                if track_id != -1:
                                    cache_matriculas[track_id] = {
                                        'rel_coords': (px1_r / w_veh, py1_r / h_veh, px2_r / w_veh, py2_r / h_veh),
                                        'frame': frame_idx,
                                        'conf': conf_matricula 
                                    }
                    else:
                        # Si el coche toca el borde, borrar su matricula de la memoria
                        if track_id != -1:
                            cache_matriculas.pop(track_id, None)

                # CAMBIO APLICADO: Preparar las variables y guardar en el CSV con el formato exacto de 15 columnas
                tiene_mat = "Si" if caja_matricula is not None else "No"
                conf_mat_str = f"{conf_matricula:.2f}" if caja_matricula is not None and conf_matricula is not None else ""
                mx1, my1, mx2, my2 = caja_matricula if caja_matricula is not None else ("", "", "", "")
                texto_matricula = "" # Queda vacio hasta la parte 2 (OCR)

                csv_writer.writerow([
                    frame_idx,
                    clase_dominante,
                    f"{confianza:.2f}",
                    track_id,
                    x1, y1, x2, y2,
                    tiene_mat,
                    conf_mat_str,
                    mx1, my1, mx2, my2,
                    texto_matricula
                ])

                # Pintar coche en el video
                color_obj = COLORES.get(clase_dominante, (0, 255, 0))
                cv2.rectangle(frame, (x1, y1), (x2, y2), color_obj, 2)
                cv2.putText(
                    frame,
                    f"ID:{track_id} {clase_dominante} {confianza:.2f}",
                    (x1, max(20, y1 - 8)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color_obj, 2
                )

                # Pintar matricula en el video
                if caja_matricula is not None:
                    cv2.rectangle(frame, (mx1, my1), (mx2, my2), COLORES['matricula'], 2)
                    cv2.putText(
                        frame, "Matricula",
                        (mx1, max(15, my1 - 5)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, COLORES['matricula'], 2
                    )

        # Cuadro oscuro con los textos (esquina superior)
        cv2.rectangle(frame, (15, 15), (270, 160), (0, 0, 0), -1)
        cv2.rectangle(frame, (15, 15), (270, 160), (255, 255, 255), 1)
        cv2.putText(frame, f"Frame: {frame_idx}/{total_frames}", (25, 38),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.52, (255, 255, 255), 1)

        y_offset = 62
        for clase, ids in conteo_unicos.items():
            color_txt = COLORES.get(clase, (200, 200, 200))
            cv2.putText(frame, f"Total {clase}s: {len(ids)}", (25, y_offset),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color_txt, 1)
            y_offset += 22

        out.write(frame)

        if frame_idx % 50 == 0 or frame_idx == total_frames:
            print(f"Progreso: {frame_idx}/{total_frames} fotogramas procesados...")

# CERRAR TODO
cap.release()
out.release()
f_csv.close()

print()
print("¡Procesamiento finalizado!")
print(f"Video generado: {VIDEO_SALIDA}")
print(f"CSV generado: {CSV_SALIDA}")
print("Totales detectados:")
for clase, ids in conteo_unicos.items():
    print(f" - {clase}: {len(ids)}")

Dispositivo asignado: cpu
Procesando 2832 fotogramas...
Progreso: 50/2832 fotogramas procesados...
Progreso: 100/2832 fotogramas procesados...
Progreso: 150/2832 fotogramas procesados...
Progreso: 200/2832 fotogramas procesados...
Progreso: 250/2832 fotogramas procesados...
Progreso: 300/2832 fotogramas procesados...
Progreso: 350/2832 fotogramas procesados...
Progreso: 400/2832 fotogramas procesados...
Progreso: 450/2832 fotogramas procesados...
Progreso: 500/2832 fotogramas procesados...
Progreso: 550/2832 fotogramas procesados...
Progreso: 600/2832 fotogramas procesados...
Progreso: 650/2832 fotogramas procesados...
Progreso: 700/2832 fotogramas procesados...
Progreso: 750/2832 fotogramas procesados...
Progreso: 800/2832 fotogramas procesados...
Progreso: 850/2832 fotogramas procesados...
Progreso: 900/2832 fotogramas procesados...
Progreso: 950/2832 fotogramas procesados...
Progreso: 1000/2832 fotogramas procesados...
Progreso: 1050/2832 fotogramas procesados...
Progreso: 1100/2832